# Spatial clustering of household WASH deprivation in Nigeria

This notebook implements the next step for Objective 2. It estimates the percentage of households deprived in at least two WASH domains and tests whether states with similar estimates cluster geographically.

1. Put this notebook beside **Nigeria_DHS_2013_2018_2024_Households_Climate.csv.gz** in your project folder.
2. Open the notebook in JupyterLab and choose **Run → Run All Cells**. If packages are missing, run **%pip install pandas numpy "scipy>=1.11" matplotlib pillow python-docx** in a notebook code cell and restart the kernel.
3. The notebook downloads a pinned Nigeria ADM1 boundary file for defining neighbours. If the download is blocked, [download it manually](https://media.githubusercontent.com/media/wmgeolab/geoBoundaries/9469f09/releaseData/gbOpen/NGA/ADM1/geoBoundaries-NGA-ADM1.geojson) and save it beside the notebook as **geoBoundaries-NGA-ADM1.geojson**. Use this full-resolution file for the neighbour calculation.
4. Open **wash_hotspot_outputs**. Automatic Word 97–2003 DOC export requires LibreOffice Writer. If the converter is unavailable, open the saved DOCX in desktop Word and save as **Word 97–2003 Document (*.doc)**.

The **main analysis uses 2018 and 2024**. The **2013 analysis is a sensitivity check** using the three-round complete-case indicator. The same boundaries are used for all rounds. Households with missing climate or GPS can still contribute.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.special import expit, logit
from scipy.stats import t

try:
    from google.colab import files
except ModuleNotFoundError:
    files = None

FILENAME = "Nigeria_DHS_2013_2018_2024_Households_Climate.csv.gz"
INPUT = Path("/content" if files else ".") / FILENAME
OUTPUT = INPUT.parent / "wash_hotspot_outputs"

if not INPUT.is_file() and files is not None:
    print(f"Choose {FILENAME}", flush=True)
    partial = INPUT.with_suffix(".gz.part")
    try:
        files.upload_file(str(partial))
        if partial.is_file() and partial.stat().st_size:
            partial.replace(INPUT)
    finally:
        partial.unlink(missing_ok=True)
if not INPUT.is_file():
    raise FileNotFoundError(f"Put {FILENAME} beside the notebook and rerun.")

KEY = ["survey_round", "hv001", "hv002"]
STRATUM = "stratum_hv023_key"
PSU = "psu_key"
DOMAINS = ["wash_dep_2plus", "wash_dep_2plus_all_rounds"]
REQUIRED = [*KEY, STRATUM, PSU, "state", "hh_weight", "hv005", *DOMAINS]
d = pd.read_csv(INPUT, usecols=REQUIRED, compression="gzip", low_memory=False)
expected = {2013: 38522, 2018: 40427, 2024: 40047}
if d.groupby("survey_round").size().to_dict() != expected:
    raise ValueError("Unexpected household totals. Check that you selected the combined household file.")
if d[KEY].isna().any().any() or d.duplicated(KEY).any():
    raise ValueError("Missing or duplicate household IDs.")
if d[[STRATUM, PSU]].isna().any().any():
    raise ValueError("A record is missing its survey stratum or PSU.")
states = sorted(d.state.dropna().unique())
if d.state.isna().any() or len(states) != 37 or "FCT" not in states:
    raise ValueError("Expected 36 states and the FCT with no missing state labels.")
for year, frame in d.groupby("survey_round"):
    if set(frame.state) != set(states):
        raise ValueError(f"{year}: state labels do not match the other rounds.")
if not np.isfinite(d.hh_weight).all() or not d.hh_weight.gt(0).all():
    raise ValueError("Household weights must be positive and finite.")
if not np.allclose(d.hh_weight, d.hv005 / 1_000_000, rtol=0, atol=1e-12):
    raise ValueError("hh_weight does not match the scaled DHS household weight hv005.")
for column in DOMAINS:
    if not d[column].dropna().isin([0, 1]).all():
        raise ValueError(f"{column} must contain only 0, 1 or a blank.")
print(f"Loaded {len(d):,} households.")


## 1. Estimate deprivation in each state

A household is eligible when its water, sanitation and hygiene outcomes are all known. The notebook uses the original household weight, survey strata and primary sampling units (PSUs). It retains all PSUs in the survey round for domain variance estimation and uses the same logit interval and single-PSU adjustment as the state descriptives notebook.

The review flags mark weighted outcome coverage below 80%, fewer than 50 eligible households, fewer than five PSUs contributing known outcomes, a 95% prevalence interval wider than 30 percentage points, or an unavailable interval. Flagged estimates enter the spatial tests and are hatched on the maps. Review flags describe the quality of the input estimate. The spatial permutation tests condition on the observed state estimates and do not propagate survey sampling error or resolve missing hygiene outcomes.


In [ ]:
rounds = dict(tuple(d.groupby("survey_round", sort=True)))
design = {}
design_rows = []
for year, frame in rounds.items():
    mapping = frame[[STRATUM, PSU]].drop_duplicates()
    if mapping[PSU].duplicated().any():
        raise ValueError(f"{year}: a PSU appears in more than one survey stratum.")
    sizes = mapping.groupby(STRATUM).size()
    df = len(mapping) - len(sizes)
    if df <= 0:
        raise ValueError(f"{year}: insufficient survey design degrees of freedom.")
    design[year] = (mapping, sizes, df)
    design_rows.append({
        "survey_round": year, "strata": len(sizes), "psus": len(mapping),
        "single_psu_strata": int(sizes.eq(1).sum()), "design_df": df,
    })

def estimate(year, state, outcome):
    frame = rounds[year]
    mapping, sizes, df = design[year]
    domain = frame.state.eq(state)
    known = domain & frame[outcome].notna()
    weights = frame.hh_weight
    count = int(domain.sum())
    n_known = int(known.sum())
    denominator = float(weights[known].sum())
    record = {
        "survey_round": year, "state": state, "indicator": outcome,
        "households_in_group": count, "unweighted_denominator": n_known,
        "unweighted_missing": count - n_known,
        "weighted_outcome_known_pct": 100 * denominator / weights[domain].sum() if count else np.nan,
        "known_psus": int(frame.loc[known, PSU].nunique()),
        "weighted_pct": np.nan, "se_pct": np.nan, "ci95_low": np.nan, "ci95_high": np.nan,
        "design_df": df, "single_psu_strata_adjusted": 0,
    }
    if not n_known:
        return record

    p = float((weights[known] * frame.loc[known, outcome]).sum() / denominator)
    record["weighted_pct"] = 100 * p
    if not 0 < p < 1:
        return record  # A logit interval is undefined at the boundaries.

    # Sum the linearised ratio residuals within each sampled PSU.
    residual = (weights[known] * (frame.loc[known, outcome] - p)).rename("residual")
    psu_totals = (
        frame.loc[known, [STRATUM, PSU]].join(residual)
        .groupby([STRATUM, PSU]).residual.sum()
        .reindex(pd.MultiIndex.from_frame(mapping), fill_value=0.0)
    )
    centred = psu_totals - psu_totals.groupby(level=0).transform("mean")
    multi = sizes[sizes.gt(1)]
    variance_by_stratum = centred.pow(2).groupby(level=0).sum().loc[multi.index]
    variance_by_stratum *= multi / (multi - 1)

    active = pd.Index(frame.loc[known, STRATUM].unique())
    lonely = int(sizes.loc[active].eq(1).sum())
    variance = float(variance_by_stratum.sum())
    if lonely:
        comparable = variance_by_stratum.loc[variance_by_stratum.index.intersection(active)]
        if comparable.empty:
            return record
        variance += lonely * float(comparable.mean())
        record["single_psu_strata_adjusted"] = lonely

    se = np.sqrt(variance) / denominator
    record["se_pct"] = 100 * se
    margin = t.ppf(0.975, df) * se / (p * (1 - p))
    record["ci95_low"], record["ci95_high"] = 100 * expit([logit(p) - margin, logit(p) + margin])
    return record


def review_flags(row):
    checks = [
        (row.unweighted_denominator < 50, "fewer_than_50_known"),
        (row.known_psus < 5, "fewer_than_5_psus"),
        (row.weighted_outcome_known_pct < 80, "coverage_below_80pct"),
        (row.ci95_high - row.ci95_low > 30, "ci_wider_than_30pp"),
        (pd.isna(row.ci95_low) or pd.isna(row.ci95_high), "ci_unavailable"),
    ]
    return " | ".join(label for flag, label in checks if flag)


outcomes = {2013: "wash_dep_2plus_all_rounds", 2018: "wash_dep_2plus", 2024: "wash_dep_2plus"}
rates = pd.DataFrame([
    estimate(year, state, outcome)
    for year, outcome in outcomes.items() for state in states
])
rates["analysis"] = np.where(rates.survey_round.eq(2013), "2013 sensitivity", "Main")
rates["review_flags"] = rates.apply(review_flags, axis=1)
if len(rates) != 111 or rates.duplicated(["survey_round", "state"]).any() or rates.weighted_pct.isna().any():
    raise ValueError("Expected 37 nonmissing state deprivation estimates per round.")
OUTPUT.mkdir(parents=True, exist_ok=True)
rates.to_csv(OUTPUT / "State_WASH_Deprivation_Prevalence.csv", index=False)
print(f"Estimated deprivation for {len(rates)} state–round combinations.")


## 2. Define neighbours and run the spatial tests

The primary neighbour graph uses Queen contiguity, defined here by shared vertices in the pinned full-resolution boundary layer. The sensitivity graph links each area to its four nearest polygon centroids by great-circle distance and makes the links symmetric. Each row of the spatial weights sums to one.

Global Moran's I tests overall spatial autocorrelation. Local Moran's I classifies high–high and low–low clusters and high–low and low–high outliers. The notebook uses **9,999 permutations** and a fixed seed. For each local test, the focal value is held fixed while values from other states are sampled without replacement. Two-sided permutation p-values measure distance from the analytical null expectation. Local tests use **Benjamini–Yekutieli FDR adjustment** across the 37 states within each round and graph. The two main global tests per graph use Holm adjustment. The 2013 global result is reported separately.

A repeated high–high classification means the state passes the local threshold in both 2018 and 2024 under the primary graph. The alternative graph is reported alongside it. This compares two survey snapshots and cannot establish what happened between surveys.


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import hashlib
import itertools
import json
import os
import shutil
import subprocess
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
from matplotlib.patches import Patch, PathPatch
from matplotlib.path import Path as PlotPath
from PIL import Image
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from scipy.stats import false_discovery_control

BOUNDARY = INPUT.parent / "geoBoundaries-NGA-ADM1.geojson"
URL = ("https://media.githubusercontent.com/media/wmgeolab/geoBoundaries/9469f09/"
       "releaseData/gbOpen/NGA/ADM1/geoBoundaries-NGA-ADM1.geojson")
CREDIT = "GRID3 Nigeria ADM1 via geoBoundaries gbOpen, 2022 boundaries, CC BY 4.0"
if not BOUNDARY.is_file():
    try:
        with urlopen(URL, timeout=45) as response:
            payload = response.read()
        json.loads(payload)
    except (OSError, ValueError) as error:
        raise RuntimeError(f"Download the boundary from the link above and save it as {BOUNDARY.name}.") from error
    BOUNDARY.write_bytes(payload)
payload = BOUNDARY.read_bytes()
geo = json.loads(payload)
if geo.get("type") != "FeatureCollection":
    raise ValueError("Expected a Nigeria ADM1 GeoJSON FeatureCollection.")
features = {}
for feature in geo["features"]:
    label = feature["properties"]["shapeName"]
    name = "FCT" if label == "Abuja Federal Capital Territory" else label
    if name in features:
        raise ValueError(f"Duplicate geometry for {name}.")
    features[name] = feature["geometry"]
if set(features) != set(states):
    raise ValueError(f"Unmatched DHS states: {sorted(set(states) - features.keys())}. "
                     f"Unmatched boundaries: {sorted(features.keys() - set(states))}.")
(OUTPUT / "Boundary_Source.txt").write_text(
    f"{CREDIT}\n{URL}\nSHA-256 {hashlib.sha256(payload).hexdigest()}\n", encoding="utf-8"
)
def polygon_path(geometry):
    kind = geometry["type"]
    if kind == "Polygon":
        polygons = [geometry["coordinates"]]
    elif kind == "MultiPolygon":
        polygons = geometry["coordinates"]
    else:
        raise ValueError(f"Unexpected geometry type: {kind}")
    paths = []
    for polygon in polygons:
        for i, ring in enumerate(polygon):
            points = np.asarray(ring, dtype=float)[:, :2]
            if len(points) < 4 or not np.isfinite(points).all():
                raise ValueError("Malformed boundary ring.")
            if not np.array_equal(points[0], points[-1]):
                points = np.vstack([points, points[0]])
            x, y = points.T
            signed_area = np.sum(x[:-1] * y[1:] - y[:-1] * x[1:])
            # Opposite winding keeps interior holes open.
            if (signed_area > 0) != (i == 0):
                points = points[::-1]
            codes = np.full(len(points), PlotPath.LINETO, dtype=np.uint8)
            codes[0], codes[-1] = PlotPath.MOVETO, PlotPath.CLOSEPOLY
            paths.append(PlotPath(points, codes))
    return PlotPath.make_compound_path(*paths)

paths = {state: polygon_path(geometry) for state, geometry in features.items()}
coordinates = np.vstack([path.vertices for path in paths.values()])
west, south = coordinates.min(axis=0)
east, north = coordinates.max(axis=0)
if not (0 < west < east < 20 and 0 < south < north < 20):
    raise ValueError("Unexpected boundary extent. Use Nigeria ADM1 longitude and latitude.")
aspect = 1 / np.cos(np.deg2rad((south + north) / 2))


def rings(path):
    starts = np.flatnonzero(path.codes == PlotPath.MOVETO)
    return np.split(path.vertices, starts[1:])

def centroid(path):
    total_area, moment = 0., np.zeros(2)
    for ring in rings(path):
        a, b = ring[:-1], ring[1:]
        cross = a[:, 0] * b[:, 1] - b[:, 0] * a[:, 1]
        total_area += cross.sum()
        moment += ((a + b) * cross[:, None]).sum(axis=0)
    return moment / (3 * total_area)

vertices = [set(map(tuple, paths[state].vertices)) for state in states]
n = len(states)
queen = np.zeros((n, n), dtype=bool)
for i, j in itertools.combinations(range(n), 2):
    queen[i, j] = queen[j, i] = bool(vertices[i] & vertices[j])

lon, lat = np.deg2rad(np.array([centroid(paths[state]) for state in states]).T)
distance = np.sin((lat[:, None] - lat) / 2)**2 + (
    np.cos(lat[:, None]) * np.cos(lat) * np.sin((lon[:, None] - lon) / 2)**2
)
np.fill_diagonal(distance, np.inf)
nearest = np.argsort(distance, axis=1)[:, :4]
knn = np.zeros_like(queen)
knn[np.arange(n)[:, None], nearest] = True
knn |= knn.T
graphs = {"Queen": queen, "Four nearest": knn}
adjacency = []
for graph, matrix in graphs.items():
    degree = matrix.sum(axis=1)
    if degree.min() == 0 or connected_components(csr_matrix(matrix), directed=False)[0] != 1:
        raise ValueError(f"{graph} has isolated areas or disconnected components. Inspect the boundary file.")
    for i, j in zip(*np.where(np.triu(matrix, 1))):
        adjacency.append({"graph": graph, "state_1": states[i], "state_2": states[j]})
    print(f"{graph}: {matrix.sum() // 2} undirected links, {degree.min()} to {degree.max()} neighbours.")
pd.DataFrame(adjacency).to_csv(OUTPUT / "State_Spatial_Neighbours.csv", index=False)

PERMUTATIONS = 9999
ALPHA = 0.05
rng = np.random.default_rng(20241109)

def permutation_p(observed, simulated, centre):
    distance = np.abs(observed - centre)
    tolerance = 1e-12 * max(1.0, distance)
    return (1 + np.count_nonzero(np.abs(simulated - centre) >= distance - tolerance)) / (len(simulated) + 1)

def spatial_tests(values, matrix):
    degree = matrix.sum(axis=1)
    w = matrix / degree[:, None]
    sd = values.std()
    if sd == 0:
        raise ValueError("A round has constant deprivation rates. Moran's I is undefined.")
    z = (values - values.mean()) / sd
    lag = w @ z
    observed_global = z @ lag / (z @ z)
    shuffled = rng.permuted(np.broadcast_to(z, (PERMUTATIONS, n)), axis=1)
    simulated_global = np.einsum("bi,ij,bj->b", shuffled, w, shuffled, optimize=True) / (z @ z)
    global_p = permutation_p(observed_global, simulated_global, -1 / (n - 1))
    local_i = z * lag
    p_values = []
    for i in range(n):
        other = np.delete(z, i)
        draws = rng.permuted(np.broadcast_to(other, (PERMUTATIONS, n - 1)), axis=1)
        simulated = z[i] * draws[:, :degree[i]].mean(axis=1)
        p_values.append(permutation_p(local_i[i], simulated, -z[i]**2 / (n - 1)))
    q = false_discovery_control(np.array(p_values), method="by")
    quadrant = np.where(z >= 0, np.where(lag >= 0, "High–high", "High–low"),
                        np.where(lag >= 0, "Low–high", "Low–low"))
    return observed_global, global_p, pd.DataFrame({
        "state": states, "neighbours": degree, "local_moran_i": local_i,
        "quadrant": quadrant, "p_permutation": p_values, "q_by": q,
        "classification": np.where(q <= ALPHA, quadrant, "Not significant"),
    })

global_rows, local_rows = [], []
for graph, matrix in graphs.items():
    for year, sample in rates.groupby("survey_round", sort=True):
        aligned = sample.set_index("state").loc[states]
        i, p_value, local = spatial_tests(aligned.weighted_pct.to_numpy(), matrix)
        global_rows.append({"graph": graph, "survey_round": year, "moran_i": i,
                            "p_permutation": p_value, "analysis": aligned.analysis.iloc[0]})
        local = local.merge(sample[["state", "weighted_pct", "review_flags"]], on="state", validate="one_to_one")
        local_rows.append(local.assign(graph=graph, survey_round=year, analysis=aligned.analysis.iloc[0]))

global_results = pd.DataFrame(global_rows)
global_results["p_holm_main_rounds"] = np.nan
for graph in graphs:
    selected = global_results.loc[global_results.graph.eq(graph) & global_results.survey_round.ne(2013), "p_permutation"]
    ordered = selected.sort_values()
    global_results.loc[ordered.index, "p_holm_main_rounds"] = np.minimum(
        1, np.maximum.accumulate(ordered.to_numpy() * np.array([2, 1]))
    )
local_results = pd.concat(local_rows, ignore_index=True)
repeated = pd.DataFrame({"state": states}).set_index("state")
for graph, prefix in [("Queen", "queen"), ("Four nearest", "nearest")]:
    subset = local_results.loc[local_results.graph.eq(graph) & local_results.survey_round.isin([2018, 2024])]
    pivot = subset.pivot(index="state", columns="survey_round", values="classification")
    for year in (2018, 2024):
        repeated[f"{prefix}_high_high_{year}"] = pivot[year].eq("High–high")
    count = repeated[[f"{prefix}_high_high_2018", f"{prefix}_high_high_2024"]].sum(axis=1)
    repeated[f"{prefix}_repeated_status"] = count.map({0: "Neither round", 1: "One round", 2: "Both rounds"})
    repeated[f"{prefix}_input_review"] = subset.assign(flag=subset.review_flags.ne("")).groupby("state").flag.any()
repeated = repeated.reset_index()
assert len(global_results) == 6 and len(local_results) == 222 and len(repeated) == 37
global_results.to_csv(OUTPUT / "State_Global_Moran_I.csv", index=False)
local_results.to_csv(OUTPUT / "State_Local_Moran_I.csv", index=False)
repeated.to_csv(OUTPUT / "State_Repeated_High_High.csv", index=False)
display(global_results.round(4))


## 3. Export the maps and Word report

The first figure compares the main survey rounds. The second shows the 2013 sensitivity analysis and the third shows repeated high–high classifications across the two main rounds. The maps display classifications from the primary Queen graph. Use the CSVs to compare the alternative graph and check which mapped areas have input review flags.

References include [Global Moran's I](https://pysal.org/esda/dev/generated/esda.Moran.html), [Local Moran's I](https://pysal.org/esda/v2.9.0/source/generated/esda.Moran_Local.html), [FDR adjustment](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.false_discovery_control.html) and the [geoBoundaries metadata](https://www.geoboundaries.org/api/current/gbOpen/NGA/ADM1/).


In [ ]:
dpi = 600

def save_figure(figure, filename):
    path = OUTPUT / filename
    figure.savefig(path, format="jpeg", dpi=dpi, facecolor="white",
                   pil_kwargs={"quality": 95, "subsampling": 0})
    plt.close(figure)
    with Image.open(path) as image:
        if image.format != "JPEG" or image.info.get("dpi") != (dpi, dpi):
            raise ValueError(f"Could not verify JPEG resolution for {path.name}.")
    return path




classes = {"High–high": "#BE4B35", "Low–low": "#28748A", "High–low": "#E6AB52",
           "Low–high": "#8CC6D6", "Not significant": "#E2E6EA"}
repeat_colors = {"Both rounds": "#A22E27", "One round": "#EFAC66", "Neither round": "#E2E6EA"}

def draw_classes(ax, frame, category, flag, colors):
    for state, path in paths.items():
        row = frame.loc[state]
        ax.add_patch(PathPatch(path, facecolor=colors[row[category]], edgecolor="#464C52",
                               linewidth=.35, hatch="////" if row[flag] else None))
    ax.set(xlim=(west - .2, east + .2), ylim=(south - .15, north + .15), aspect=aspect)
    ax.set_axis_off()

def class_key(fig, colors):
    handles = [Patch(facecolor=color, edgecolor="#464C52", label=label) for label, color in colors.items()]
    handles.append(Patch(facecolor="white", edgecolor="#464C52", hatch="////", label="Input estimate flagged"))
    fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(.5, .07),
               ncol=3, frameon=False, fontsize=8)
    fig.text(.5, .025, CREDIT, ha="center", fontsize=6.5)

queen_results = local_results.loc[local_results.graph.eq("Queen")].assign(
    flagged=lambda x: x.review_flags.ne("")
)
fig, axes = plt.subplots(1, 2, figsize=(7, 3.65))
fig.subplots_adjust(left=.035, right=.965, top=.78, bottom=.24, wspace=.07)
fig.suptitle("Local WASH deprivation clusters", y=.95, fontsize=13, fontweight="bold")
for ax, year in zip(axes, (2018, 2024)):
    draw_classes(ax, queen_results.loc[queen_results.survey_round.eq(year)].set_index("state"),
                 "classification", "flagged", classes)
    ax.set_title(str(year), fontsize=10)
class_key(fig, classes)
main_map = save_figure(fig, "Map_WASH_Local_Clusters_2018_2024_600dpi.jpg")

fig, ax = plt.subplots(figsize=(7, 3.45))
fig.subplots_adjust(top=.85, bottom=.25)
fig.suptitle("Local WASH deprivation clusters in 2013", y=.96, fontsize=13, fontweight="bold")
draw_classes(ax, queen_results.loc[queen_results.survey_round.eq(2013)].set_index("state"),
             "classification", "flagged", classes)
class_key(fig, classes)
sensitivity_map = save_figure(fig, "Map_WASH_Local_Clusters_2013_Sensitivity_600dpi.jpg")

fig, ax = plt.subplots(figsize=(7, 3.6))
fig.subplots_adjust(top=.85, bottom=.24)
fig.suptitle("Repeated high–high classification", y=.96, fontsize=13, fontweight="bold")
draw_classes(ax, repeated.set_index("state"), "queen_repeated_status", "queen_input_review", repeat_colors)
class_key(fig, repeat_colors)
repeat_map = save_figure(fig, "Map_WASH_Repeated_High_High_600dpi.jpg")
from docx import Document
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Inches, Pt, RGBColor

figure_width = 7
doc = Document()
doc.core_properties.title = "Spatial clustering of household WASH deprivation in Nigeria"
doc.core_properties.author = ""
section = doc.sections[0]
section.page_width, section.page_height = Inches(8.5), Inches(11)
section.left_margin = section.right_margin = Inches(0.75)
section.top_margin, section.bottom_margin = Inches(0.7), Inches(0.7)

for style_name, size in [("Normal", 10.5), ("Title", 25), ("Heading 1", 17), ("Heading 2", 13), ("Caption", 9)]:
    style = doc.styles[style_name]
    style.font.name = "Calibri"
    style.font.size = Pt(size)
    style.font.color.rgb = RGBColor.from_string("000000")
    style.paragraph_format.space_before = Pt(5)
    style.paragraph_format.space_after = Pt(5)
    if style_name == "Title":
        border = style.element.get_or_add_pPr().find(qn("w:pBdr"))
        if border is not None:
            border.getparent().remove(border)

def xml(parent, tag, **attributes):
    element = OxmlElement(f"w:{tag}")
    for key, value in attributes.items():
        element.set(qn(f"w:{key}"), str(value))
    parent.append(element)
    return element

def add_table(caption, headers, rows, widths, text_columns, font_size=9.5):
    doc.add_paragraph(caption, style="Caption").paragraph_format.keep_with_next = True
    table = doc.add_table(rows=0, cols=len(headers))
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    table.autofit = False
    for column, width in zip(table.columns, widths):
        column.width = Inches(width)
    props = table._tbl.tblPr
    borders = xml(props, "tblBorders")
    for edge in ("top", "left", "bottom", "right", "insideH", "insideV"):
        xml(borders, edge, val="single", sz=4, color="D9D9D9")
    margins = xml(props, "tblCellMar")
    for edge, value in (("top", 16), ("bottom", 16), ("left", 85), ("right", 85)):
        xml(margins, edge, w=value, type="dxa")

    for i, values in enumerate([headers, *rows]):
        row = table.add_row()
        row_props = row._tr.get_or_add_trPr()
        xml(row_props, "cantSplit")
        if i == 0:
            xml(row_props, "tblHeader")
        for j, (cell, value, width) in enumerate(zip(row.cells, values, widths)):
            cell.width = Inches(width)
            cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER
            if i == 0 or i % 2 == 0:
                xml(cell._tc.get_or_add_tcPr(), "shd", fill="203A50" if i == 0 else "F2F6F9")
            paragraph = cell.paragraphs[0]
            paragraph.alignment = WD_ALIGN_PARAGRAPH.LEFT if j in text_columns else WD_ALIGN_PARAGRAPH.CENTER
            paragraph.paragraph_format.space_after = Pt(0)
            paragraph.paragraph_format.space_before = Pt(0)
            paragraph.paragraph_format.line_spacing = 1.05
            run = paragraph.add_run(str(value))
            run.bold = i == 0
            run.font.size = Pt(font_size)
            if i == 0:
                run.font.color.rgb = RGBColor.from_string("FFFFFF")
    return table

def add_figure(path, caption, description):
    paragraph = doc.add_paragraph()
    paragraph.paragraph_format.space_before = Pt(7)
    paragraph.paragraph_format.keep_with_next = True
    paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
    shape = paragraph.add_run().add_picture(str(path), width=Inches(figure_width))
    shape._inline.docPr.set("descr", description)
    doc.add_paragraph(caption, style="Caption")



doc.add_paragraph("Spatial clustering of household WASH deprivation in Nigeria", style="Title")
doc.add_paragraph("Nigeria Demographic and Health Survey rounds 2013, 2018 and 2024")
doc.add_paragraph(
    "Deprivation means two or more WASH domains below basic among households with all three domains known. "
    "The 2018 and 2024 surveys form the main analysis. The 2013 estimate is a complete-case sensitivity analysis."
)
doc.add_heading("Global spatial autocorrelation", level=1)
def probability(value):
    return f"{value:.4f}" if pd.notna(value) else "—"

global_table = [[r.graph, r.survey_round, f"{r.moran_i:.3f}", probability(r.p_permutation),
                 probability(r.p_holm_main_rounds)] for r in global_results.itertuples(index=False)]
add_table("Table 1. Global Moran's I by survey round and neighbour graph",
          ["Neighbours", "Round", "Moran's I", "Permutation p", "Holm p"], global_table,
          [1.7, .8, 1.1, 1.7, 1.7], {0})
doc.add_heading("High high classifications", level=1)
hh_table = []
for r in global_results.itertuples(index=False):
    mask = local_results.graph.eq(r.graph) & local_results.survey_round.eq(r.survey_round)
    high = local_results.loc[mask & local_results.classification.eq("High–high"), "state"].tolist()
    hh_table.append([r.graph, r.survey_round, len(high), ", ".join(high) if high else "None detected"])
add_table("Table 2. High–high classifications after local FDR adjustment",
          ["Neighbours", "Round", "Count", "States"], hh_table, [1.45, .75, .75, 4.05], {0, 3})
for graph, prefix in [("Queen", "queen"), ("Four nearest", "nearest")]:
    names = repeated.loc[repeated[f"{prefix}_repeated_status"].eq("Both rounds"), "state"].tolist()
    text = ", ".join(names) + "." if names else "None detected."
    doc.add_paragraph(f"High–high in both main rounds under {graph} weights. {text}")
doc.add_paragraph(
    "Permutation results are conditional on the estimated state prevalences. "
    "Hatching identifies inputs to review for missing outcomes or low precision. "
    "The maps use the same 2022 GRID3 boundaries for all three survey rounds."
)
doc.add_heading("Main rounds and repeated clusters", level=1).paragraph_format.page_break_before = True
add_figure(main_map, "Figure 1. Local Moran classifications for 2018 and 2024.",
           "Two Nigeria maps showing adjusted local Moran classifications.")
add_figure(repeat_map, "Figure 2. Repeated high–high status under Queen weights.",
           "Map of areas classified high–high in neither, one, or both main survey rounds.")
doc.add_heading("Sensitivity analysis for 2013", level=1).paragraph_format.page_break_before = True
add_figure(sensitivity_map, "Figure 3. Local Moran classifications for 2013.",
           "Nigeria map showing adjusted local Moran classifications in the 2013 sensitivity analysis.")
doc.add_paragraph(
    "Queen neighbours share vertices in the full-resolution boundaries. The alternative graph symmetrises "
    "links to the four nearest centroids. The spatial tests use 9,999 permutations. "
    "Local p-values are adjusted across the 37 areas within each round and graph. "
    "The global Holm adjustment covers the two main rounds separately for each graph."
)
def find_soffice():
    custom = os.environ.get("LIBREOFFICE_PATH")
    if custom and Path(custom).is_file():
        return custom
    candidates = [shutil.which("soffice"), shutil.which("libreoffice"),
                  "/Applications/LibreOffice.app/Contents/MacOS/soffice"]
    for variable in ("ProgramFiles", "ProgramFiles(x86)"):
        if base := os.environ.get(variable):
            candidates.append(str(Path(base) / "LibreOffice" / "program" / "soffice.com"))
    return next((candidate for candidate in candidates if candidate and Path(candidate).is_file()), None)


docx_path = OUTPUT / "Nigeria_DHS_WASH_Hotspots_Report.docx"
doc.save(docx_path)

def export_legacy_word(source, soffice):
    with TemporaryDirectory() as scratch:
        temp = Path(scratch).resolve()
        result = subprocess.run(
            [soffice, f"-env:UserInstallation={(temp / 'profile').as_uri()}", "--headless",
             "--convert-to", "doc:MS Word 97", "--outdir", str(temp), str(source.resolve())],
            capture_output=True, text=True, timeout=180,
        )
        converted = temp / source.with_suffix(".doc").name
        if result.returncode or not converted.is_file():
            raise RuntimeError(result.stderr or result.stdout or "Converter returned no file.")
        with converted.open("rb") as stream:
            if stream.read(8) != bytes.fromhex("D0CF11E0A1B11AE1"):
                raise ValueError("The converter did not create a binary Word 97–2003 document.")
        shutil.copyfile(converted, source.with_suffix(".doc"))

soffice = find_soffice()
if soffice:
    try:
        export_legacy_word(docx_path, soffice)
        print(f"Saved {docx_path.with_suffix('.doc').resolve()}")
    except (RuntimeError, ValueError, OSError, subprocess.TimeoutExpired) as error:
        print(f"Word .doc conversion failed: {error}")
        print("Open the saved .docx in desktop Word and save it as Word 97–2003 Document (*.doc).")
else:
    print("LibreOffice was unavailable. Open the saved .docx in desktop Word and save it as Word 97–2003 Document (*.doc).")


print(f"Saved the report, JPEG maps and CSV results in {OUTPUT.resolve()}.")
